In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load raw telemetry data to initiate data quality assessment.

df = pd.read_json('../data/raw/llm_telemetry_v1_2026-05-26.json')
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,gpt-3.5-turbo,1136.0,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,gemini_1.5,340.0,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133.0,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,gpt-4,469.0,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,GPT-4,757.0,861,2241.32,200,aws-us-west


In [3]:
df.columns

Index(['timestamp', 'session_id', 'model_name', 'input_tokens',
       'output_tokens', 'latency_ms', 'status_code', 'gpu_cluster'],
      dtype='str')

In [4]:
df.shape

(1030, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      1030 non-null   datetime64[us]
 1   session_id     1030 non-null   str           
 2   model_name     1030 non-null   str           
 3   input_tokens   952 non-null    float64       
 4   output_tokens  1030 non-null   int64         
 5   latency_ms     1030 non-null   object        
 6   status_code    1030 non-null   int64         
 7   gpu_cluster    984 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(2), object(1), str(3)
memory usage: 64.5+ KB


In [6]:
df.isna().sum()

timestamp         0
session_id        0
model_name        0
input_tokens     78
output_tokens     0
latency_ms        0
status_code       0
gpu_cluster      46
dtype: int64

In [7]:
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,gpt-3.5-turbo,1136.0,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,gemini_1.5,340.0,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133.0,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,gpt-4,469.0,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,GPT-4,757.0,861,2241.32,200,aws-us-west


In [8]:
# Standardize model nomenclature by removing whitespace and applying title casing for consistent grouping.

df['model_name'] = df['model_name'].apply(lambda x: x.strip().title())
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136.0,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340.0,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133.0,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469.0,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757.0,861,2241.32,200,aws-us-west


In [9]:
# Impute missing input_token values with the median to maintain statistical integrity.

median_input_tokens = df['input_tokens'].median()
df.fillna({'input_tokens' : median_input_tokens},inplace= True)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136.0,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340.0,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133.0,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469.0,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757.0,861,2241.32,200,aws-us-west


In [10]:
# Convert token counts to integer format to enable precise numerical operations.

df['input_tokens'] = df['input_tokens'].astype(int)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [11]:
# Verify dataset completeness by quantifying remaining null values post-imputation.

df.isna().sum()

timestamp         0
session_id        0
model_name        0
input_tokens      0
output_tokens     0
latency_ms        0
status_code       0
gpu_cluster      46
dtype: int64

In [12]:
# Standardize latency metrics by coercing non-numeric anomalies to NaN for subsequent cleaning.

df['latency_ms'] = pd.to_numeric(df['latency_ms'], errors= 'coerce')
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      1030 non-null   datetime64[us]
 1   session_id     1030 non-null   str           
 2   model_name     1030 non-null   str           
 3   input_tokens   1030 non-null   int64         
 4   output_tokens  1030 non-null   int64         
 5   latency_ms     1004 non-null   float64       
 6   status_code    1030 non-null   int64         
 7   gpu_cluster    984 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 64.5 KB


In [14]:
df.isna().sum()

timestamp         0
session_id        0
model_name        0
input_tokens      0
output_tokens     0
latency_ms       26
status_code       0
gpu_cluster      46
dtype: int64

In [15]:
# Resolve latency outliers by imputing median values to ensure representative performance analysis.

median_latency_ms = df['latency_ms'].median()
df.fillna({'latency_ms' : median_latency_ms}, inplace= True)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,NaN
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      1030 non-null   datetime64[us]
 1   session_id     1030 non-null   str           
 2   model_name     1030 non-null   str           
 3   input_tokens   1030 non-null   int64         
 4   output_tokens  1030 non-null   int64         
 5   latency_ms     1030 non-null   float64       
 6   status_code    1030 non-null   int64         
 7   gpu_cluster    984 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 64.5 KB


In [17]:
# Categorize missing infrastructure metadata as 'Unknown' to preserve session-level log data.

df.fillna({'gpu_cluster': "Unknown"}, inplace= True)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [18]:
df.isna().sum()

timestamp        0
session_id       0
model_name       0
input_tokens     0
output_tokens    0
latency_ms       0
status_code      0
gpu_cluster      0
dtype: int64

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      1030 non-null   datetime64[us]
 1   session_id     1030 non-null   str           
 2   model_name     1030 non-null   str           
 3   input_tokens   1030 non-null   int64         
 4   output_tokens  1030 non-null   int64         
 5   latency_ms     1030 non-null   float64       
 6   status_code    1030 non-null   int64         
 7   gpu_cluster    1030 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 64.5 KB


In [20]:
# Persist cleaned telemetry dataset to CSV for downstream EDA.

df.to_csv('../data/processed/llm_telemetry_v1_cleaned_2026-05-26.csv', index= False)